## To-Do List

1. **Assign IDs to `gallery_df`**
   - For each gallery detection, match it with the ground-truth bounding boxes using **IoU**.
   - Assign the corresponding **ground-truth person ID** based on the best IoU match.

2. **Evaluate Query–Gallery Matches**
   - For each query image, compare its **ground-truth ID** with the **IDs of the retrieved gallery images**.
   - Mark retrieved results as **positive** if the IDs match, otherwise **negative**.

3. **Compute Evaluation Metrics**
   - **mAP (mean Average Precision)**:
     - For each query, compute **AP** as the average of precision values at ranks where a positive match is retrieved.
     - Report **mAP** as the mean AP over all queries.
   - **Rank-1 Accuracy**:
     - Compute the fraction of queries for which the **top-1 retrieved result** is a positive match.

4. **Train and Evaluate with Metric Learning**
   - Use **`frame_train`** to train a metric learning model.
   - Use the **test set** to perform retrieval for query images and evaluate performance using mAP and Rank-1 accuracy.

### Imports and Configs

In [4]:
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 947.7 kB/s eta 0:00:0000:0100:01


In [2]:
# lets get started with single image
image_path = "../archive/frames"
import os
# sample random image from the folder
import random
from IPython.display import display
from PIL import Image
import time
from ultralytics import YOLO
import pandas as pd
import torch
from torchvision import transforms
import matplotlib.pyplot as plt
import cv2
from PIL import ImageDraw
%matplotlib inline
import numpy as np
from tqdm import tqdm
import torch.nn.functional as F
import faiss




In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EMB_DIM = 2048
IMG_SIZE = (256, 128)
TOP_K = 10


In [4]:

# To reload the index and metadata later:
index = faiss.read_index("gallery_with_resnet50.index")
gallery_df = pd.read_csv("gallery_metadata_with_resnet50.csv")

### Load models

In [41]:
detector = YOLO("yolov8n.pt").to(DEVICE).eval()
reid_model = torch.hub.load('pytorch/vision:v0.10.0', "resnet50", weights="ResNet50_Weights.DEFAULT" ).to(DEVICE).eval()
reid_model.fc = torch.nn.Identity()  # Remove the final classification layer
reid_model = reid_model.to(DEVICE).eval()




Using cache found in /home/saad/.cache/torch/hub/pytorch_vision_v0.10.0


In [42]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

### Index image paths for gallery and query images

In [69]:
def index_images(root):
    return [
        os.path.join(root, f)
        for root, _, files in os.walk(root)
        for f in files
        if f.endswith((".jpg", ".png"))
    ]

gallery_paths = index_images("../archive/frames")
gallery_paths = gallery_paths # Debug purpose
query_paths = index_images("../archive/query_box")
query_paths = query_paths[:10]



In [70]:

IMG_SIZE = (224, 224)  # Example size, adjust as needed

def detect_and_crop(img, detector, target_class=0):
    """
    Detect objects in the image and crop all instances of the target class (default: person).

    Args:
        img (np.ndarray): Input image.
        detector (YOLO): YOLO object detector.
        target_class (int or list): Class ID(s) to keep. Default 0 (person)

    Returns:
        crops (list): List of cropped images.
        det_info (list): List of detection info dictionaries.
    """
    detections = detector(img, verbose=False)
    crops = []
    det_info = []

    for det in detections:
        num_boxes = len(det.boxes)
        for i in range(num_boxes):
            bbox = det.boxes.xyxy[i].cpu().numpy()
            conf = det.boxes.conf[i].cpu().numpy()
            cls = int(det.boxes.cls[i].cpu().numpy())

            # Only keep target class(es)
            if (isinstance(target_class, int) and cls != target_class) or \
               (isinstance(target_class, list) and cls not in target_class):
                continue

            # Save detection info
            det_info.append({
                "x_min": bbox[0],
                "y_min": bbox[1],
                "x_max": bbox[2],
                "y_max": bbox[3],
                "confidence": conf,
                "class": cls,
            })

            # Crop and resize
            crop = img[int(bbox[1]):int(bbox[3]), int(bbox[0]):int(bbox[2])]
            if crop.size == 0:
                continue
            crop = cv2.resize(crop, IMG_SIZE)
            crops.append(crop)

    return crops, det_info


In [71]:
def extract_embeddings(crops, model):

    """Extract embeddings from cropped images using the re-identification model.
    Args:
        crops (list): List of cropped images.
        model (torch.nn.Module): The re-identification model.
    Returns:
        emb (numpy.ndarray): Array of embeddings.
    """

    # Ensure batch dimension and consistent array type
    if isinstance(crops, np.ndarray) and crops.ndim == 3:
        crops = crops[None, ...]  # add batch dim for single image
    elif isinstance(crops, list):
        crops = np.stack(crops, axis=0)  # list -> (B, H, W, C)

    crops = torch.tensor(crops).permute(0,3,1,2).float().to(DEVICE) / 255.
    with torch.no_grad():
        emb = model(crops)
        if isinstance(emb, (list, tuple)):
            emb = emb[0]
        emb = F.normalize(emb, dim=1)
    return emb.cpu().numpy()


In [72]:
columns = [
    "image_path",
    "x_min", "y_min", "x_max", "y_max",
    "confidence",
    "class",
    "faiss_id"
]

gallery_rows = []

index = faiss.IndexFlatIP(EMB_DIM)


### Build gallery (single loop, clean)

In [74]:
for path in tqdm(gallery_paths):
    img = cv2.imread(path)
    if img is None:
        continue

    crops, det_info = detect_and_crop(img, detector)

    if len(crops) == 0:
        continue

    embeddings = extract_embeddings(crops, reid_model)
    # print(embeddings.shape)

    index.add(embeddings)
    faiss_ids = np.arange(index.ntotal - len(embeddings), index.ntotal)

    for info, fid in zip(det_info, faiss_ids):
        gallery_rows.append([
            path,
            info["x_min"],
            info["y_min"],
            info["x_max"],
            info["y_max"],
            info["confidence"],
            info["class"],
            fid
        ])


100%|██████████| 11816/11816 [1:40:56<00:00,  1.95it/s] 


### Save the fais index

In [ ]:
faiss.write_index(index, "gallery_with_resnet50.index")
gallery_df = pd.DataFrame(gallery_rows, columns=columns)
gallery_df.to_csv("gallery_metadata_with_resnet50.csv", index=False)


# To reload the index and metadata later:
# index = faiss.read_index("gallery_with_resnet50.index")
# gallery_df = pd.read_csv("gallery_metadata_with_resnet50.csv")


In [5]:
from scipy.io import loadmat
import os

person_ids = []
mat_cache = {}

for _, row in gallery_df.iterrows():
    path = row['image_path']
    filename = os.path.splitext(os.path.basename(path))[0]
    mat_path = os.path.join('../archive/annotations/', filename + '.jpg.mat')

    # Load .mat once per image
    if mat_path not in mat_cache:
        try:
            data = loadmat(mat_path)
            box_new = data['box_new']
            mat_cache[mat_path] = {
                (b[1], b[2], b[3], b[4]): int(b[0])
                for b in box_new
            }
        except Exception:
            mat_cache[mat_path] = {}

    x_min, y_min, x_max, y_max = row['x_min'], row['y_min'], row['x_max'], row['y_max']
    w, h = x_max - x_min, y_max - y_min

    person_ids.append(
        mat_cache[mat_path].get((x_min, y_min, w, h), -1)
    )

gallery_df['person_id'] = person_ids


In [7]:
gallery_df["person_id"].value_counts()

person_id
-1    52925
Name: count, dtype: int64

### Query pipeline

In [84]:
def query_image(path, reid_model, index, gallery_df, k=10):
    img = cv2.imread(path)
    img = cv2.resize(img, IMG_SIZE)

    emb = extract_embeddings([img], reid_model)
    D, I = index.search(emb, k)   # -> D: distances, I: indices

    results = gallery_df.iloc[I[0]].copy()
    results["similarity"] = D[0]
    return results


In [103]:
results = query_image(query_paths[2], reid_model, index, gallery_df) 


In [104]:
query_paths[2]

'../archive/query_box/931_c5s2_041530.jpg'

In [105]:
results_sorted = results.sort_values("similarity", ascending=False)
display(results_sorted.head(TOP_K))

,image_path,x_min,y_min,x_max,y_max,confidence,class,faiss_id,similarity
32352,../archive/frames/c5s2_041530.jpg,394.722015,350.080475,473.012238,523.472473,0.83259326,0,32352,0.889262
44106,../archive/frames/c5s2_041555.jpg,456.927338,341.332428,522.549011,511.021118,0.80067044,0,44106,0.811540
1875,../archive/frames/c5s1_095773.jpg,629.958984,356.560364,710.874390,543.856934,0.56241286,0,1875,0.786250
34023,../archive/frames/c5s1_094973.jpg,998.140503,341.094055,1069.043823,511.457794,0.6753126,0,34023,0.760412
21240,../archive/frames/c5s2_008705.jpg,450.054321,352.434082,517.221252,539.747681,0.7988301,0,21240,0.756637
3359,../archive/frames/c4s4_027535.jpg,1793.462036,283.862915,1849.039673,449.172455,0.7648045,0,3359,0.753982
51055,../archive/frames/c5s2_107802.jpg,768.146729,333.097412,835.950500,509.401062,0.7591617,0,51055,0.753522
37356,../archive/frames/c3s2_081428.jpg,1756.803955,419.920471,1815.906738,597.869202,0.6655261,0,37356,0.751371
34730,../archive/frames/c5s1_061248.jpg,382.993652,358.449097,446.367828,531.237854,0.7263012,0,34730,0.750421
10003,../archive/frames/c5s2_037230.jpg,1127.977417,311.208862,1189.536255,477.327301,0.6416273,0,10003,0.750160


In [108]:
gallery_df

,image_path,x_min,y_min,x_max,y_max,confidence,class,faiss_id
0,../archive/frames/c5s1_067023.jpg,1696.411011,298.038849,1835.033936,579.928406,0.8369659,0,0
1,../archive/frames/c5s1_067023.jpg,1158.350220,332.113464,1214.699097,495.463531,0.7382743,0,1
2,../archive/frames/c5s1_067023.jpg,355.437286,343.482239,438.214508,517.090393,0.7248444,0,2
3,../archive/frames/c5s1_067023.jpg,475.557068,353.948212,512.741394,436.783661,0.7022781,0,3
4,../archive/frames/c5s1_067023.jpg,771.640015,338.984070,820.678589,453.393768,0.5346871,0,4
...,...,...,...,...,...,...,...,...
52920,../archive/frames/c1s1_048101.jpg,23.347492,495.793884,132.929886,681.925720,0.46127746,0,52920
52921,../archive/frames/c1s4_005556.jpg,1474.673584,442.670288,1711.028687,1075.301880,0.88600075,0,52921
52922,../archive/frames/c1s4_005556.jpg,108.053925,414.000092,348.162903,1075.435425,0.81917816,0,52922
52923,../archive/frames/c1s4_005556.jpg,480.861603,476.803925,535.917297,579.574402,0.41679108,0,52923


In [111]:
from scipy.io import loadmat
data = loadmat('../archive/annotations/c5s1_067023.jpg.mat')
print(data["box_new"])

[[        206      473.98      343.54      44.348      96.522]
 [        233      346.15      335.72      86.087      190.43]
 [        248      772.72       325.9      48.985      130.63]
 [        254      1152.9      323.57      67.646      177.28]
 [        258      1694.1      293.24      153.95      296.24]]
